# Heart Disease Prediction Using Deep Learning

This notebook is the **Google Colab training workflow** for the final project. It uses the same shared project code from `heart_model.py` and trains the teacher-required models:

- Feedforward ANN for multi-class classification
- Dropout and L2 regularization when TensorFlow/Keras is available
- Logistic Regression baseline
- Random Forest comparison model
- Low Risk / Medium Risk / High Risk output

The Streamlit app should load the saved model artifact from the `model/` folder instead of retraining every time.

## 1. Colab Setup

Run this cell in Google Colab. Upload these files when prompted:

- `heart_model.py`
- `processed.cleveland.data`

If you already uploaded them to the Colab working directory, you can skip the upload prompt.

In [ ]:
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    needed = ["heart_model.py", "processed.cleveland.data"]
    missing = [name for name in needed if not Path(name).exists()]
    if missing:
        print("Upload these files:", missing)
        files.upload()
    else:
        print("Required files already found in Colab.")
else:
    print("Not running in Colab. Using local files.")

## 2. Install / Check Dependencies

In [ ]:
if IN_COLAB:
    %pip install -q joblib scikit-learn pandas numpy matplotlib seaborn tensorflow

import tensorflow as tf
print("TensorFlow version:", tf.__version__)

## 3. Import Shared Project Code

In [ ]:
import shutil
import sqlite3

import pandas as pd
from IPython.display import display

from heart_model import (
    DATA_FILE,
    DB_FILE,
    DISPLAY_COLUMNS,
    FEATURES,
    KERAS_MODEL_FILE,
    METRICS_FILE,
    MODEL_DIR,
    PREPROCESSOR_FILE,
    RISK_LABELS,
    load_dataset,
    load_metrics,
    load_trained_model,
    save_model_artifacts,
    tensorflow_available,
)

print("TensorFlow available to project code:", tensorflow_available())

## 4. Load Dataset

This uses the same loader as the Streamlit app, so columns, preprocessing assumptions, and risk-class mapping stay consistent.

In [ ]:
df = load_dataset(add_patient_id=True)
print("Dataset file:", DATA_FILE)
print("Dataset shape:", df.shape)
display(df[DISPLAY_COLUMNS].head())

print("Risk class counts:")
display(df["risk_class"].value_counts().sort_index().rename(index=RISK_LABELS).to_frame("records"))

## 5. Train Once And Save Artifacts

In Colab, this trains the Keras ANN path with Dropout and L2 regularization, then compares it against Logistic Regression and Random Forest. The selected project artifact is saved in the `model/` folder.

In [ ]:
metrics = save_model_artifacts()

print("Selected model:", metrics["selected_model"])
print("Artifact type:", metrics["artifact_type"])
print("Model file:", metrics["model_file"])
print("Metrics file:", METRICS_FILE)
print(f"Accuracy: {metrics['accuracy']:.4f}")
print("\nTraining note:")
print(metrics["training_note"])

print("\nModel comparison:")
display(pd.DataFrame(metrics["comparison"]))

print("\nClassification report:")
print(metrics["classification_report_text"])

## 6. Verify Saved Model Inference

This mirrors the Streamlit app: load the saved artifact, prepare one patient record, and call `predict_proba`.

In [ ]:
model = load_trained_model()

sample_input = {
    "age": 54,
    "sex": 1,
    "cp": 4,
    "trestbps": 130,
    "chol": 246,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.2,
    "slope": 1,
    "ca": 0,
    "thal": 3,
}

input_df = pd.DataFrame([sample_input], columns=FEATURES)
probabilities = model.predict_proba(input_df)[0]
predicted_class = int(probabilities.argmax())

print("Predicted risk:", RISK_LABELS[predicted_class])
display(pd.DataFrame({
    "risk_level": list(RISK_LABELS.values()),
    "probability": probabilities,
}))

## 7. Export Model Folder For Streamlit

Run this after training. It creates `model_artifacts_for_streamlit.zip`. Download it, unzip it locally, and replace the `model/` folder inside your `Application` project.

In [ ]:
zip_path = shutil.make_archive("model_artifacts_for_streamlit", "zip", MODEL_DIR)
print("Created:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print("Zip file is ready locally.")

## 8. Optional: View Prediction History Database

The Streamlit app stores saved predictions in SQLite. This cell is only for viewing the database if it exists in the notebook environment.

In [ ]:
if DB_FILE.exists():
    with sqlite3.connect(DB_FILE) as conn:
        history_df = pd.read_sql_query(
            "SELECT * FROM predictions ORDER BY id DESC LIMIT 20",
            conn,
        )
    if history_df.empty:
        print("Prediction database exists, but no predictions have been saved yet.")
    else:
        display(history_df)
else:
    print("Prediction database not found in this environment.")